# PostgreSQL — Databricks Integration

Populate and read SLED relational data from PostgreSQL via JDBC and Lakehouse Federation.

**SLED tables (auto-created on populate):**

| Use Case | Tables |
|---|---|
| `student_enrollment` | sled_students, sled_courses, sled_enrollment_events |
| `grant_budget` | sled_budget_transactions, sled_agencies, sled_vendors |
| `citizen_services` | sled_service_requests, sled_citizens, sled_assets |
| `k12_early_warning` | sled_k12_events, sled_k12_students, sled_schools |
| `procurement` | sled_procurement_events, sled_contracts |
| `case_management` | sled_case_events, sled_clients, sled_cases |

**Other tables:** `kafka_event_logs` (Kafka event audit log)

## Prerequisites
- Data API Collector stack running
- Network access from Databricks to your host (port 15433 for Postgres SSL)
- PostgreSQL JDBC driver on cluster (pre-installed on Databricks)
- Databricks secrets configured

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — UPDATE THESE for your environment
# ---------------------------------------------------------------------------
HOST = "your-hostname.com"

# Secrets — loaded from Databricks secret scope
# Set up with:
#   databricks secrets create-scope data-api
#   databricks secrets put-secret data-api api-key --string-value "YOUR_SECRET_KEY"
#   databricks secrets put-secret data-api kafka-user --string-value "YOUR_KAFKA_USER"
#   databricks secrets put-secret data-api kafka-password --string-value "YOUR_KAFKA_PASSWORD"
#   databricks secrets put-secret data-api neo4j-password --string-value "YOUR_NEO4J_PASSWORD"
#   databricks secrets put-secret data-api postgres-password --string-value "YOUR_POSTGRES_PASSWORD"

API_KEY           = dbutils.secrets.get(scope="data-api", key="api-key")
KAFKA_USER        = dbutils.secrets.get(scope="data-api", key="kafka-user")
KAFKA_PASSWORD    = dbutils.secrets.get(scope="data-api", key="kafka-password")
NEO4J_USER        = "neo4j"
NEO4J_PASSWORD    = dbutils.secrets.get(scope="data-api", key="neo4j-password")
POSTGRES_USER     = "postgres"
POSTGRES_PASSWORD = dbutils.secrets.get(scope="data-api", key="postgres-password")

# Derived URLs
API_URL         = f"https://{HOST}/api/v1"
KAFKA_BROKER    = f"{HOST}:9093"
NEO4J_URI       = f"bolt+s://{HOST}:7688"
POSTGRES_JDBC   = f"jdbc:postgresql://{HOST}:15433/datacollector?ssl=true&sslmode=require"
HEADERS         = {"X-Api-Key": API_KEY}

# Checkpoint base path — update to your catalog/schema
CHECKPOINT_BASE = "/Volumes/your_catalog/your_schema/checkpoints"

# SLED use cases
SLED_USE_CASES = [
    "student_enrollment", "grant_budget", "citizen_services",
    "k12_early_warning", "procurement", "case_management",
]

# Kafka connection options (reusable)
KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": KAFKA_BROKER,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{KAFKA_USER}" '
        f'password="{KAFKA_PASSWORD}";'
    ),
    "startingOffsets": "earliest",
}

# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------
import requests
import builtins
from pyspark.sql.types import *
from pyspark.sql.functions import *


def read_kafka_stream(topic, schema):
    """Read a Kafka topic as a parsed Structured Streaming DataFrame."""
    raw = spark.readStream.format("kafka")
    for k, v in KAFKA_OPTIONS.items():
        raw = raw.option(k, v)
    raw = raw.option("subscribe", topic).load()
    return (
        raw
        .select(from_json(col("value").cast("string"), schema).alias("data"))
        .select("data.*")
    )


def read_pg_table(table_name):
    """Read a table from PostgreSQL via JDBC."""
    return (
        spark.read
        .format("jdbc")
        .option("url", POSTGRES_JDBC)
        .option("dbtable", table_name)
        .option("user", POSTGRES_USER)
        .option("password", POSTGRES_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .load()
    )


print(f"Config loaded — API: {API_URL}, Kafka: {KAFKA_BROKER}")


---
## Populate PostgreSQL with SLED Data

Generate relational data for all 6 use cases. Tables are created automatically.

In [ ]:
for use_case in SLED_USE_CASES:
    resp = requests.post(
        f"{API_URL}/data-sources/sled/{use_case}/populate",
        headers=HEADERS,
        json={"num_records": 5000},
    )
    data = resp.json()
    print(f"{use_case}: job={data.get('job_id')} status={data.get('status')}")

In [ ]:
# Check populate status
import time
time.sleep(15)

for use_case in SLED_USE_CASES:
    status = requests.get(f"{API_URL}/data-sources/sled/{use_case}/status", headers=HEADERS).json()
    total = builtins.sum(v for v in status["counts"].values() if isinstance(v, int))
    print(f"{use_case}: {total} rows")

---
## Core — Kafka Event Logs

In [ ]:
df_events = read_pg_table("kafka_event_logs")
print(f"Kafka event logs: {df_events.count():,} rows")
display(df_events.limit(20))

---
## Student Enrollment

In [ ]:
df_students = read_pg_table("sled_students")
df_courses = read_pg_table("sled_courses")
df_enrollment = read_pg_table("sled_enrollment_events")

print(f"Students: {df_students.count():,}")
print(f"Courses: {df_courses.count():,}")
print(f"Enrollment events: {df_enrollment.count():,}")

In [ ]:
# Enrollments per semester
display(
    df_enrollment
    .filter(col("action") == "enroll")
    .groupBy("semester")
    .agg(
        count("*").alias("enrollments"),
        countDistinct("student_id").alias("unique_students"),
    )
    .orderBy("semester")
)

In [ ]:
# Student distribution by status and campus
display(
    df_students
    .groupBy("status", "campus")
    .agg(
        count("*").alias("students"),
        avg("gpa").alias("avg_gpa"),
        avg("total_credits").alias("avg_credits"),
    )
    .orderBy(col("students").desc())
)

---
## Grant & Budget

In [ ]:
df_budget = read_pg_table("sled_budget_transactions")
df_agencies = read_pg_table("sled_agencies")
df_vendors = read_pg_table("sled_vendors")

print(f"Transactions: {df_budget.count():,}")
print(f"Agencies: {df_agencies.count():,}")
print(f"Vendors: {df_vendors.count():,}")

In [ ]:
# Expenditures by fund category and fiscal year
display(
    df_budget
    .filter(col("transaction_type") == "expenditure")
    .groupBy("fund_category", "fiscal_year")
    .agg(
        count("*").alias("transactions"),
        sum("amount").alias("total_spent"),
    )
    .orderBy("fiscal_year", col("total_spent").desc())
)

In [ ]:
# Vendor diversity analysis
display(
    df_vendors
    .groupBy("type")
    .agg(
        count("*").alias("vendors"),
        avg(col("minority_owned").cast("int")).alias("pct_minority"),
        avg(col("small_business").cast("int")).alias("pct_small_biz"),
        avg(col("woman_owned").cast("int")).alias("pct_woman_owned"),
        avg("annual_revenue").alias("avg_revenue"),
    )
    .orderBy(col("vendors").desc())
)

---
## Citizen Services (311)

In [ ]:
df_requests = read_pg_table("sled_service_requests")
df_citizens = read_pg_table("sled_citizens")
df_assets = read_pg_table("sled_assets")

print(f"Service requests: {df_requests.count():,}")
print(f"Citizens: {df_citizens.count():,}")
print(f"Assets: {df_assets.count():,}")

In [ ]:
# Request resolution analysis
display(
    df_requests
    .groupBy("request_type", "priority")
    .agg(
        count("*").alias("requests"),
        avg("response_time_hours").alias("avg_response_hours"),
        avg("satisfaction_rating").alias("avg_satisfaction"),
    )
    .orderBy(col("requests").desc())
)

In [ ]:
# Asset condition by district
display(
    df_assets
    .groupBy("district", "condition")
    .agg(
        count("*").alias("assets"),
        avg("replacement_cost").alias("avg_replacement_cost"),
    )
    .orderBy("district", "condition")
)

---
## K-12 Early Warning

In [ ]:
df_k12 = read_pg_table("sled_k12_students")
df_schools = read_pg_table("sled_schools")
df_k12_events = read_pg_table("sled_k12_events")

print(f"Students: {df_k12.count():,}")
print(f"Schools: {df_schools.count():,}")
print(f"Events: {df_k12_events.count():,}")

In [ ]:
# At-risk students by school
display(
    df_k12
    .filter(col("risk_score") > 70)
    .join(df_schools, "school_id")
    .groupBy("name", "type")
    .agg(
        count("*").alias("high_risk_students"),
        avg("risk_score").alias("avg_risk_score"),
        avg("attendance_rate").alias("avg_attendance"),
    )
    .orderBy(col("high_risk_students").desc())
)

In [ ]:
# Demographic risk analysis
display(
    df_k12
    .groupBy("grade_level")
    .agg(
        count("*").alias("students"),
        avg("risk_score").alias("avg_risk"),
        avg("gpa").alias("avg_gpa"),
        avg(col("free_reduced_lunch").cast("int")).alias("pct_frl"),
        avg(col("english_learner").cast("int")).alias("pct_ell"),
        avg(col("special_education").cast("int")).alias("pct_sped"),
    )
    .orderBy("grade_level")
)

---
## Procurement

In [ ]:
df_contracts = read_pg_table("sled_contracts")
df_proc_events = read_pg_table("sled_procurement_events")

print(f"Contracts: {df_contracts.count():,}")
print(f"Procurement events: {df_proc_events.count():,}")

In [ ]:
# Contract analysis by category and method
display(
    df_contracts
    .groupBy("category", "method")
    .agg(
        count("*").alias("contracts"),
        sum("amount").alias("total_value"),
        avg("amount").alias("avg_value"),
        avg("duration_months").alias("avg_duration"),
    )
    .orderBy(col("total_value").desc())
)

---
## Case Management (HHS)

In [ ]:
df_clients = read_pg_table("sled_clients")
df_cases = read_pg_table("sled_cases")
df_case_events = read_pg_table("sled_case_events")

print(f"Clients: {df_clients.count():,}")
print(f"Cases: {df_cases.count():,}")
print(f"Case events: {df_case_events.count():,}")

In [ ]:
# Benefit analysis by program
display(
    df_cases
    .groupBy("program", "determination")
    .agg(
        count("*").alias("cases"),
        avg("benefit_amount").alias("avg_benefit"),
        sum("benefit_amount").alias("total_benefits"),
    )
    .orderBy(col("total_benefits").desc())
)

In [ ]:
# Client demographics
display(
    df_clients
    .groupBy("income_bracket", "primary_language")
    .agg(
        count("*").alias("clients"),
        avg("household_size").alias("avg_household"),
        avg(col("veteran").cast("int")).alias("pct_veteran"),
        avg(col("disabled").cast("int")).alias("pct_disabled"),
    )
    .orderBy(col("clients").desc())
)

---
## Lakehouse Federation (Optional)

Use Unity Catalog to query PostgreSQL directly without JDBC code.

```sql
-- Create a foreign connection
CREATE CONNECTION postgres_conn
TYPE POSTGRESQL
OPTIONS (
  host '<your-host>',
  port '15433',
  user 'dataapi',
  password secret('postgres', 'password')
);

-- Create a foreign catalog
CREATE FOREIGN CATALOG postgres_data
USING CONNECTION postgres_conn
OPTIONS (database 'data_collector');

-- Query SLED tables directly
SELECT * FROM postgres_data.public.sled_students LIMIT 10;
SELECT * FROM postgres_data.public.sled_k12_students WHERE risk_score > 70;
```

---
## Cleanup

In [ ]:
# Uncomment to clear all SLED Postgres data
# for use_case in SLED_USE_CASES:
#     resp = requests.delete(f"{API_URL}/data-sources/sled/{use_case}/clear", headers=HEADERS)
#     print(f"{use_case}: {resp.json()}")